# Deliverable 3 — Factorized collision compilation audit

Compiles the reusable R block, validates the R/Q/Kronecker matvec, and compares it with the full dense dilation. A complete factorized PREPARE/SELECT circuit is still not claimed.

In [1]:
from pathlib import Path
import sys
repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
import json, numpy as np, pandas as pd
from collections import Counter
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
from quantum_aero.carleman import operators, unitary_dilation
from quantum_aero.deliverables import sparse_collision_oracle
from quantum_aero.quantum import transpiled_collision_resources

omega=1.2
oracle=sparse_collision_oracle(omega)
L,Q=operators(); R=(1-omega)*np.eye(9)+omega*L
UR,alpha_r,padded_r=unitary_dilation(R)
qc=QuantumCircuit(int(np.log2(len(UR)))); qc.append(UnitaryGate(UR), range(qc.num_qubits))
compiled=transpile(qc,basis_gates=["u","cx"],optimization_level=0,seed_transpiler=7)
r_resource={"qubits":qc.num_qubits,"alpha":alpha_r,"native_depth":compiled.depth(),"operations":dict(Counter(compiled.count_ops()))}
r_resource, oracle

({'qubits': 5,
  'alpha': 1.6810249675906652,
  'native_depth': 825,
  'operations': {'u': 752, 'cx': 423}},
 {'dimension': 90,
  'nnz': 7138,
  'density': 0.8812345679012346,
  'max_row_sparsity': 81,
  'median_row_sparsity': 81.0,
  'unique_coefficient_magnitudes': 35,
  'oracle_matvec_max_error': 8.881784197001252e-16,
  'factorized_matvec_max_error': 1.4432899320127035e-15,
  'r_nnz': 81,
  'q_nnz': 496,
  'factorized_stored_coefficients': 577,
  'flat_to_factorized_storage_ratio': 12.370883882149046,
  'column_index_bits': 7,
  'row_count_bits': 7})

In [3]:
dense=transpiled_collision_resources(omega)
rotation_t_cost=int(np.ceil(3*np.log2(1/1e-10)+10))
comparison=pd.DataFrame([
 {"implementation":"dense 256x256 dilation","qubits":dense["logical_qubits"],"native_depth":dense["transpiled_depth"],"cx":dense["transpiled_operations"].get("cx",0),"rotation_T_proxy":dense["transpiled_operations"].get("u",0)*rotation_t_cost},
 {"implementation":"compiled reusable R dilation","qubits":r_resource["qubits"],"native_depth":r_resource["native_depth"],"cx":r_resource["operations"].get("cx",0),"rotation_T_proxy":r_resource["operations"].get("u",0)*rotation_t_cost},
])
comparison

,implementation,qubits,native_depth,cx,rotation_T_proxy
0,dense 256x256 dilation,8,107126,87216,5663240
1,compiled reusable R dilation,5,825,423,82720


In [4]:
payload={"factorized_validation":oracle,"compiled_R_block":r_resource,"dense_reference":dense,"comparison":comparison.to_dict("records"),
"remaining_blocker":"compile Q loading and coherent R/Q composition as PREPARE/SELECT; R-only compilation is not a full collision block encoding"}
(output_dir/"08_factorized_collision_compilation.json").write_text(json.dumps(payload,indent=2))
assert oracle["factorized_matvec_max_error"]<1e-12
print("PASS with boundary: factorized arithmetic validated and R compiled; full PREPARE/SELECT remains open.")

PASS with boundary: factorized arithmetic validated and R compiled; full PREPARE/SELECT remains open.
